In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/omr-data-preprocessing


/home/dongmin/userdata/dongmin/omr-data-preprocessing/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import groupby

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21
import musicdiff as md
import converter21 as c21

import xml.etree.ElementTree as ET
from modules.lmxe.lmxe import linearize_lmxe, delinearize_lmxe
from modules.lmxe.lmxe.utils import split_tokens_by_key_token, find_nth_token_index
from modules.lmxe.lmxe.Linearizer import LMXELinearizer
from modules.lmxe.lmxe.Delinearizer import LMXEDelinearizer
from modules.lmxe.lmxe.LMXEFile import LMXEFile, LMXEMetadata, get_score_type
from modules.lmxe.lmxe.symbolic.MxlFile import MusicXMLFile
from modules.lmxe.lmxe.symbolic.part_to_score import parts_to_score
from modules.lmxe.lmxe.symbolic.debug_compare import compare_parts

from omrdp.utils import load_ossq_metadata
from omrdp.utils import parse_log_and_get_exclusions, print_exclusion_report
from omrdp.utils.image import crop_white_space

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
  return

## Load metadata

In [3]:
dataset_dir = Path.home() / 'userdata' / 'open-score-string-quartets'
dataset_data_dir = dataset_dir / 'data'
dataset_score_dir = dataset_dir / 'scores'

dataset_dir.exists()

True

In [4]:
metadata = load_ossq_metadata(dataset_data_dir / 'scores.yaml')
len(metadata)

122

In [5]:
temp_dir = Path.cwd() / 'temp'
temp_dir.mkdir(exist_ok=True)

## MusicXML to ABC partwise

In [6]:
fatal_error_set = {
  'conversion error',
  'measure insert/delete', 
  'staff insert/delete',
  'staff group',
  'barline',
  'keysig', 'clef', 'tie',
  'note', 'dot',
}

### Synthetic

In [7]:
abc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/split_abc_partwise_synthetic_2026-04-03-16:53:05.log'

eabc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/abc_to_eabc_synthetic_partwise_2026-04-03-17:15:07.log'

excluded_segments_path = dataset_dir / 'excluded_segments_synthetic_abc_partwise.txt'

In [8]:
with open(dataset_dir / 'partwise_abc_alignment_synthetic.txt', 'r') as f:
  reader = csv.reader(f)
  pw_alignment = list(reader)

pw_alignment = {
  row[0]: row[1]
  for row in pw_alignment
}

In [9]:
abc_log_file = Path(abc_log_file)
(
  abc_excluded_segments, 
  abc_num_samples, 
  abc_known_errors, 
  abc_known_fatal_errors, 
  abc_error_cnt, 
  abc_failed_cnt
) = parse_log_and_get_exclusions(
  abc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

abc_excluded_segments = [
  pw_alignment[s_name]
  for s_name in abc_excluded_segments
]

print(f"ABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=abc_num_samples,
  excluded_segments=abc_excluded_segments,
  known_errors=abc_known_errors,
  known_fatal_errors=abc_known_fatal_errors,
  error_cnt=abc_error_cnt,
  failed_cnt=abc_failed_cnt,
)

ABC Conversion Exclusion Report:
total: 52952
excluded segments: 2066
exclusion rate: 3.9016467744372263

error segments: 2062
failed segments: 4

Known fatal errors:
{
  "clef": 1276,
  "tie": 483,
  "note": 24268,
  "measure insert/delete": 3647,
  "keysig": 272,
  "dot": 1029
}

Known errors:
{
  "accidental": 989,
  "slur": 23342,
  "clef": 1276,
  "tie": 483,
  "note": 24268,
  "measure insert/delete": 3647,
  "keysig": 272,
  "flag/beam": 947,
  "note head": 406,
  "dot": 1029,
  "fingered tremolo": 297,
  "ornament": 336,
  "multi-staff arpeggio": 159,
  "articulation": 1
}


In [10]:
eabc_log_file = Path(eabc_log_file)
(
  eabc_excluded_segments, 
  eabc_num_samples, 
  eabc_known_errors, 
  eabc_known_fatal_errors, 
  eabc_error_cnt, 
  eabc_failed_cnt
) = parse_log_and_get_exclusions(
  eabc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"EABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=eabc_num_samples,
  excluded_segments=eabc_excluded_segments,
  known_errors=eabc_known_errors,
  known_fatal_errors=eabc_known_fatal_errors,
  error_cnt=eabc_error_cnt,
  failed_cnt=eabc_failed_cnt,
)

EABC Conversion Exclusion Report:
total: 52949
excluded segments: 1
exclusion rate: 0.0018886097943303934

error segments: 0
failed segments: 1

Known fatal errors:
{}

Known errors:
{}


In [12]:
abc_excluded_set = set(abc_excluded_segments)
eabc_excluded_set = set(eabc_excluded_segments)

print("len abc:", len(abc_excluded_set))
print("len eabc:", len(eabc_excluded_set))
print("len intersection:", len(abc_excluded_set & eabc_excluded_set))
print("len union:", len(abc_excluded_set | eabc_excluded_set))
print("len abc - eabc:", len(abc_excluded_set - eabc_excluded_set))
print("len eabc - abc:", len(eabc_excluded_set - abc_excluded_set))

len abc: 2066
len eabc: 1
len intersection: 0
len union: 2067
len abc - eabc: 2066
len eabc - abc: 1


In [13]:
excluded_list = list(abc_excluded_set | eabc_excluded_set)
print(f"Total Segments: {abc_num_samples}")
print(f"Total Excluded Segments: {len(excluded_list)}")
print(f"Total Exclusion Rate: {len(excluded_list) / abc_num_samples * 100:.2f}%")

Total Segments: 52952
Total Excluded Segments: 2067
Total Exclusion Rate: 3.90%


In [14]:
with open(excluded_segments_path, 'w') as f:
  for seg_id in excluded_list:
    f.write(seg_id + '\n')

### Scanned

In [15]:
abc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/split_abc_partwise_scanned_2026-04-03-17:00:36.log'

eabc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/abc_to_eabc_scanned_partwise_2026-04-03-17:35:23.log'

excluded_segments_path = dataset_dir / 'excluded_segments_scanned_abc_partwise.txt'

In [16]:
with open(dataset_dir / 'partwise_abc_alignment_scanned.txt', 'r') as f:
  reader = csv.reader(f)
  pw_alignment = list(reader)

pw_alignment = {
  row[0]: row[1]
  for row in pw_alignment
}

In [17]:
abc_log_file = Path(abc_log_file)
(
  abc_excluded_segments, 
  abc_num_samples, 
  abc_known_errors, 
  abc_known_fatal_errors, 
  abc_error_cnt, 
  abc_failed_cnt
) = parse_log_and_get_exclusions(
  abc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

abc_excluded_segments = [
  pw_alignment[s_name]
  for s_name in abc_excluded_segments
]

print(f"ABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=abc_num_samples,
  excluded_segments=abc_excluded_segments,
  known_errors=abc_known_errors,
  known_fatal_errors=abc_known_fatal_errors,
  error_cnt=abc_error_cnt,
  failed_cnt=abc_failed_cnt,
)

ABC Conversion Exclusion Report:
total: 45208
excluded segments: 1815
exclusion rate: 4.0147761458149

error segments: 1811
failed segments: 4

Known fatal errors:
{
  "clef": 1097,
  "measure insert/delete": 3608,
  "note": 22802,
  "tie": 436,
  "keysig": 244,
  "dot": 1005
}

Known errors:
{
  "slur": 19977,
  "accidental": 826,
  "clef": 1097,
  "measure insert/delete": 3608,
  "note": 22802,
  "tie": 436,
  "keysig": 244,
  "flag/beam": 911,
  "note head": 360,
  "dot": 1005,
  "fingered tremolo": 279,
  "ornament": 261,
  "multi-staff arpeggio": 105,
  "articulation": 1
}


In [19]:
eabc_log_file = Path(eabc_log_file)
(
  eabc_excluded_segments, 
  eabc_num_samples, 
  eabc_known_errors, 
  eabc_known_fatal_errors, 
  eabc_error_cnt, 
  eabc_failed_cnt
) = parse_log_and_get_exclusions(
  eabc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"EABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=eabc_num_samples,
  excluded_segments=eabc_excluded_segments,
  known_errors=eabc_known_errors,
  known_fatal_errors=eabc_known_fatal_errors,
  error_cnt=eabc_error_cnt,
  failed_cnt=eabc_failed_cnt,
)

EABC Conversion Exclusion Report:
total: 45202
excluded segments: 2
exclusion rate: 0.004424582983053847

error segments: 0
failed segments: 2

Known fatal errors:
{}

Known errors:
{}


In [21]:
abc_excluded_set = set(abc_excluded_segments)
eabc_excluded_set = set(eabc_excluded_segments)

print("len kern:", len(abc_excluded_set))
print("len ekern:", len(eabc_excluded_set))
print("len intersection:", len(abc_excluded_set & eabc_excluded_set))
print("len union:", len(abc_excluded_set | eabc_excluded_set))
print("len abc - eabc:", len(abc_excluded_set - eabc_excluded_set))
print("len eabc - abc:", len(eabc_excluded_set - abc_excluded_set))

len kern: 1815
len ekern: 1
len intersection: 0
len union: 1816
len abc - eabc: 1815
len eabc - abc: 1


In [22]:
excluded_list = list(abc_excluded_set | eabc_excluded_set)
print(f"Total Segments: {abc_num_samples}")
print(f"Total Excluded Segments: {len(excluded_list)}")
print(f"Total Exclusion Rate: {len(excluded_list) / abc_num_samples * 100:.2f}%")

Total Segments: 45208
Total Excluded Segments: 1816
Total Exclusion Rate: 4.02%


In [23]:
with open(excluded_segments_path, 'w') as f:
  for seg_id in excluded_list:
    f.write(seg_id + '\n')